# Orbit — LoRA fine-tune Gemma 3 270M

Before running:
1. Runtime → Change runtime type → **GPU** (T4 is fine).
2. Upload to Google Drive, keeping this layout:
   - `MyDrive/orbit/lora_fine_tuning_orbit_model/` — `script.py`, `orbit_schema.py`, `generate.py`, `export_litert.py`, `orbit_samples.jsonl` (no `.env`, `.venv` or `runs`)
   - `MyDrive/orbit/dataset/` — `orbit_bubbles_v3.train.jsonl`, `orbit_bubbles_v2.clean.val.jsonl`

Everything the notebook produces is written straight to `MyDrive/orbit/runs/`, so a disconnect loses nothing.
If the runtime drops during training, rerun the setup cell and the training cell: it resumes from the last finished epoch.

The model is `unsloth/gemma-3-270m-it`, an ungated copy of Google's Gemma 3 270M IT weights, so no Hugging Face token is needed.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    print('No HF_TOKEN secret; downloading anonymously')
DRIVE = '/content/drive/MyDrive/orbit'
RUNS = f'{DRIVE}/runs'
os.makedirs(RUNS, exist_ok=True)
# code and data are copied locally for speed; outputs go to RUNS on Drive
!rm -rf /content/orbit && mkdir -p /content/orbit && cp -r {DRIVE}/lora_fine_tuning_orbit_model {DRIVE}/dataset /content/orbit/
%cd /content/orbit/lora_fine_tuning_orbit_model
!ls ../dataset {RUNS}

In [ ]:
!pip install -q -U transformers peft accelerate

## Train (LoRA, ~40 min on a T4)
Checkpoints are saved to Drive after every epoch. The best epoch by validation loss is kept, then the script generates on the 200 validation entries and writes `eval.json`.

In [ ]:
!python script.py --train ../dataset/orbit_bubbles_v3.train.jsonl --val ../dataset/orbit_bubbles_v2.clean.val.jsonl --out {RUNS}/orbit-lora-v3 --epochs 5 --batch 4 --grad-accum 4 --lr 2e-4 --rank 32 --model unsloth/gemma-3-270m-it

In [ ]:
import json
RUN = f'{RUNS}/orbit-lora-v3'   # change to f'{RUNS}/orbit-full' to inspect the full fine-tune
metrics = json.load(open(f'{RUN}/eval.json'))
print(json.dumps({k: v for k, v in metrics.items() if k not in ('failures', 'predictions')}, indent=2))
for failure in metrics['failures'][:10]:
    print('---', failure['problem'])
    print(failure['entry'])
    print('>>', failure['reply'])
    print()

## Export to `.litertlm`
Run this right after training. The file lands in `MyDrive/orbit/runs/orbit-lora-v3/litert/`.

In [ ]:
!pip install -q litert-torch
!python export_litert.py --merged {RUNS}/orbit-lora-v3/merged --out {RUNS}/orbit-lora-v3/litert

## Try your own entry

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from orbit_schema import user_message, repair_bubbles
import json
MERGED = f'{RUNS}/orbit-lora-v3/merged'
tok = AutoTokenizer.from_pretrained(MERGED)
model = AutoModelForCausalLM.from_pretrained(MERGED, dtype=torch.float32).to('cuda')
entry = "long day at work and my back hurts. gotta call grandma tmrw its her birthday"
prompt = tok.apply_chat_template([{'role': 'user', 'content': user_message(entry)}], tokenize=False, add_generation_prompt=True)
ids = tok(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
out = model.generate(**ids, max_new_tokens=512, max_length=None, do_sample=False)
reply = tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True)
print(reply)
try:
    print('after repair:', repair_bubbles(entry, json.loads(reply)['bubbles']))
except Exception as error:
    print('reply was not JSON:', error)

## Optional: full fine-tune for comparison (~40 min)
Trains all 270M weights instead of a LoRA. Compare its `eval.json` with the LoRA run (set `RUN` in the metrics cell), and if it wins, export it with `--merged {RUNS}/orbit-full/merged --out {RUNS}/orbit-full/litert`.

In [ ]:
!python script.py --full --train ../dataset/orbit_bubbles_v3.train.jsonl --val ../dataset/orbit_bubbles_v2.clean.val.jsonl --out {RUNS}/orbit-full --epochs 4 --batch 4 --grad-accum 4 --lr 5e-5 --model unsloth/gemma-3-270m-it